# WordPiece tokenization

Install the Transformers, Datasets, and Evaluate libraries to run this notebook.

In [1]:
%%capture
!pip install datasets evaluate transformers[sentencepiece]

Also log into Hugging Face.

In [2]:
from huggingface_hub import notebook_login

notebook_login()

WordPiece is the <font color='blue'>tokenization algorithm</font> Google developed to <font color='blue'>pretrain BERT</font>. It has since been reused in quite a few <font color='blue'>Transformer models</font> based on BERT, such as <font color='blue'>DistilBERT</font>, <font color='blue'>MobileBERT</font>, <font color='blue'>Funnel Transformers</font>, and <font color='blue'>MPNET</font>. It's very similar to BPE in terms of the training, but the actual <font color='blue'>tokenization</font> is <font color='blue'>done differently</font>.


## Training algorithm

<Tip warning={true}>

⚠️ Google never open-sourced its implementation of the training algorithm of WordPiece, so what follows is a best guess based on the published literature. It may not be 100% accurate.

</Tip>

Like BPE, WordPiece starts from a <font color='blue'>small vocabulary</font> including the <font color='blue'>special tokens</font> used by the <font color='blue'>model</font> and the <font color='blue'>initial alphabet</font>. Since it identifies <font color='blue'>subwords</font> by <font color='blue'>adding</font> a <font color='blue'>prefix</font> (like `##` for BERT), each <font color='blue'>word</font> is initially <font color='blue'>split</font> by <font color='blue'>adding that prefix</font> to <font color='blue'>all</font> the <font color='blue'>characters</font> inside the <font color='blue'>word</font>. So, for instance, `"word"` gets split like this:

```
w ##o ##r ##d
```

Thus, the <font color='blue'>initial alphabet</font> contains <font color='blue'>all</font> the <font color='blue'>characters</font> present at the <font color='blue'>beginning</font> of a word and the characters present <font color='blue'>inside a word</font> preceded by the WordPiece prefix.

Then, again like BPE, WordPiece learns <font color='blue'>merge rules</font>. The <font color='blue'>main difference</font> is the <font color='blue'>way</font> the <font color='blue'>pair to be merged</font> is <font color='blue'>selected</font>. Instead of selecting the most frequent pair, WordPiece <font color='blue'>computes a score</font> for <font color='blue'>each pair</font>, using the following formula:

$$\mathrm{score} = \frac{\mathrm{freq\_of\_pair}}{ \mathrm{freq\_of\_first\_element} \times \mathrm{freq\_of\_second\_element}}$$

By <font color='blue'>dividing</font> the <font color='blue'>frequency of the pair</font> by the <font color='blue'>product of the frequencies of each of its parts</font>, the algorithm <font color='blue'>prioritizes</font> the merging of pairs where the <font color='blue'>individual parts are less frequent</font> in the vocabulary. For instance, it <font color='blue'>won't</font> necessarily <font color='blue'>merge `("un", "##able")`</font> even if that pair <font color='blue'>occurs very frequently</font> in the <font color='blue'>vocabulary</font>, because the two pairs `"un"` and `"##able"` will likely each appear in a lot of other words and have a high frequency. In contrast, a pair like <font color='blue'>`("hu", "##gging")`</font> will probably be <font color='blue'>merged faster</font> (assuming the word "hugging" appears often in the vocabulary) since <font color='blue'>`"hu"`</font> and <font color='blue'>`"##gging"`</font> are likely to be <font color='blue'>less frequent</font> individually.

Let's look at the same vocabulary we used in the BPE training example:

```
("hug", 10), ("pug", 5), ("pun", 12), ("bun", 4), ("hugs", 5)
```

The splits here will be:

```
("h" "##u" "##g", 10), ("p" "##u" "##g", 5), ("p" "##u" "##n", 12), ("b" "##u" "##n", 4), ("h" "##u" "##g" "##s", 5)
```

so the initial vocabulary will be `["b", "h", "p", "##g", "##n", "##s", "##u"]` (if we forget about special tokens for now). The most frequent pair is <font color='blue'>`("##u", "##g")`</font> (present <font color='blue'>20</font> times), but the individual frequency of `"##u"` is very high, so its score is not the highest (it's <font color='blue'>1 / 36</font>). All pairs with a <font color='blue'>`"##u"`</font> actually have that <font color='blue'>same score (1 / 36)</font>, so the best score goes to the pair <font color='blue'>`("##g", "##s")`</font> -- the only one without a `"##u"` -- at <font color='blue'>1 / 20</font>, and the <font color='blue'>first merge learned</font> is <font color='blue'>`("##g", "##s") -> ("##gs")`</font>.

Note that when we <font color='blue'>merge</font>, we <font color='blue'>remove</font> the <font color='blue'>`##` between</font> the <font color='blue'>two tokens</font>, so we <font color='blue'>add `"##gs"`</font> to the <font color='blue'>vocabulary</font> and <font color='blue'>apply the merge</font> in the words of the corpus:

```
Vocabulary: ["b", "h", "p", "##g", "##n", "##s", "##u", "##gs"]
Corpus: ("h" "##u" "##g", 10), ("p" "##u" "##g", 5), ("p" "##u" "##n", 12), ("b" "##u" "##n", 4), ("h" "##u" "##gs", 5)
```

At this point, <font color='blue'>`"##u"`</font> is in <font color='blue'>all</font> the <font color='blue'>possible pairs</font>, so they <font color='blue'>all</font> end up with the <font color='blue'>same score</font>. Let's say that in this case, the first pair is merged, so <font color='blue'>`("h", "##u") -> "hu"`</font>. This takes us to:

```
Vocabulary: ["b", "h", "p", "##g", "##n", "##s", "##u", "##gs", "hu"]
Corpus: ("hu" "##g", 10), ("p" "##u" "##g", 5), ("p" "##u" "##n", 12), ("b" "##u" "##n", 4), ("hu" "##gs", 5)
```

Then the next best score is shared by <font color='blue'>`("hu", "##g")` and `("hu", "##gs")`</font> (with <font color='blue'>1/15</font>, compared to <font color='blue'>1/21</font> for all the other pairs), so the <font color='blue'>first pair</font> with the <font color='blue'>biggest score</font> is <font color='blue'>merged</font>:

```
Vocabulary: ["b", "h", "p", "##g", "##n", "##s", "##u", "##gs", "hu", "hug"]
Corpus: ("hug", 10), ("p" "##u" "##g", 5), ("p" "##u" "##n", 12), ("b" "##u" "##n", 4), ("hu" "##gs", 5)
```

and we continue like this until we reach the desired vocabulary size.

The next merge rule will be <font color='blue'>("hu", "##gs") -> "hugs"</font>.
Among the remaining pairs, ("hu", "##gs") has the highest score of 1/15, compared to other pairs like ("p", "##u"), ("##u", "##g"), and ("##u", "##n") which all have lower scores of 1/21 due to the high individual frequency of "##u". The corpus becomes:

```
Vocabulary: ["b", "h", "p", "##g", "##n", "##s", "##u", "##gs", "hu", "hug", "hugs"]
Corpus: ("hug", 10), ("p" "##u" "##g", 5), ("p" "##u" "##n", 12), ("b" "##u" "##n", 4), ("hugs", 5)
```

## Tokenization algorithm
Tokenization differs in WordPiece and BPE in that <font color='blue'>WordPiece</font> only <font color='blue'>saves</font> the <font color='blue'>final vocabulary</font>, <font color='blue'>not</font> the <font color='blue'>merge rules</font> learned. Starting from the word to tokenize, <font color='blue'>WordPiece</font> finds the <font color='blue'>longest subword</font> that is <font color='blue'>in</font> the <font color='blue'>vocabulary</font>, then splits on it. For instance, if we use the vocabulary learned in the example above, for the word <font color='blue'>`"hugs"`</font> the <font color='blue'>longest subword</font> starting from the beginning that is inside the vocabulary is <font color='blue'>`"hug"`</font>, so we <font color='blue'>split there</font> and get <font color='blue'>`["hug", "##s"]`</font>. We then <font color='blue'>continue</font> with <font color='blue'> `"##s"`</font>, which is in the vocabulary, so the <font color='blue'>tokenization</font> of <font color='blue'>`"hugs"`</font> is <font color='blue'>`["hug", "##s"]`</font>.

With <font color='blue'>BPE</font>, we would have applied the merges learned in order and <font color='blue'>tokenized</font> this <font color='blue'>as `["hu", "##gs"]`</font>, so the encoding is different.

As <font color='blue'>another example</font>, let's see how the word <font color='blue'>`"bugs"`</font> would be <font color='blue'>tokenized</font>. <font color='blue'>`"b"`</font> is the <font color='blue'>longest subword</font> starting at the beginning of the word that is in the vocabulary, so we split there and get <font color='blue'>`["b", "##ugs"]`</font>. Then <font color='blue'>`"##u"`</font> is the <font color='blue'>longest subword</font> starting at the <font color='blue'>beginning</font> of <font color='blue'>`"##ugs"`</font> that is <font color='blue'>in</font> the <font color='blue'>vocabulary</font>, so we split there and get <font color='blue'>`["b", "##u, "##gs"]`</font>. Finally, `"##gs"` is in the vocabulary, so this last list is the tokenization of `"bugs"`.

When the tokenization gets to a stage where it's <font color='blue'>not possible</font> to find a <font color='blue'>subword</font> in the <font color='blue'>vocabulary</font>, the <font color='blue'>whole word</font> is <font color='blue'>tokenized</font> as <font color='blue'>unknown</font> -- so, for instance, <font color='blue'>`"mug"`</font> would be <font color='blue'>tokenized</font> as <font color='blue'>`["[UNK]"]`</font>, as would <font color='blue'>`"bum"`</font> (even if we can begin with `"b"` and `"##u"`, `"##m"` is not the vocabulary, and the resulting tokenization will just be `["[UNK]"]`, not `["b", "##u", "[UNK]"]`). This is another difference from <font color='blue'>BPE</font>, which would <font color='blue'>only classify</font> the <font color='blue'>individual characters</font> not in the vocabulary as <font color='blue'>unknown</font>.


**Example:** How will the word `"pugs"` be tokenized?


For the word <font color='blue'>"pugs"</font>, we start by finding the <font color='blue'>longest subword</font> from the <font color='blue'>beginning</font> that is in the vocabulary. <font color='blue'>"p"</font> is the longest subword starting at the beginning of the word that is in the vocabulary, so we <font color='blue'>split there</font> and get <font color='blue'>["p", "##ugs"]</font>. Then <font color='blue'>"##u"</font> is the <font color='blue'>longest subword</font> starting at the <font color='blue'>beginning</font> of <font color='blue'>"##ugs"</font> that is in the vocabulary, so we <font color='blue'>split there</font> and get <font color='blue'>["p", "##u", "##gs"]</font>. Then, since <font color='blue'>"##gs"</font> is <font color='blue'>in</font> the <font color='blue'>vocabulary</font>, the tokenization of "pugs" is ["p", "##u", "##gs"].

## Implementing WordPiece

Now let's take a look at an <font color='blue'>implementation</font> of the <font color='blue'>WordPiece algorithm</font>. Like with BPE, this is just pedagogical, and you won't able to use this on a big corpus.

We will use the same corpus as in the BPE example:

In [17]:
corpus = [
    "This is the Hugging Face Course.",
    "This chapter is about tokenization.",
    "This section shows several tokenizer algorithms.",
    "Hopefully, you will be able to understand how they are trained and generate tokens.",
]

<font color='blue'></font>We need to <font color='blue'>pre-tokenize</font> the <font color='blue'>corpus</font> into <font color='blue'>words</font>. Since we are replicating a WordPiece tokenizer (like BERT), we will use the `bert-base-cased` tokenizer for the pre-tokenization:

In [18]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("bert-base-cased")

Then we <font color='blue'>compute</font> the <font color='blue'>frequencies</font> of <font color='blue'>each word</font> in the corpus as we do the pre-tokenization:

In [19]:
from collections import defaultdict

word_freqs = defaultdict(int)

for text in corpus:
    words_with_offsets = tokenizer.backend_tokenizer.pre_tokenizer.pre_tokenize_str(text)
    print(words_with_offsets)

[('This', (0, 4)), ('is', (5, 7)), ('the', (8, 11)), ('Hugging', (12, 19)), ('Face', (20, 24)), ('Course', (25, 31)), ('.', (31, 32))]
[('This', (0, 4)), ('chapter', (5, 12)), ('is', (13, 15)), ('about', (16, 21)), ('tokenization', (22, 34)), ('.', (34, 35))]
[('This', (0, 4)), ('section', (5, 12)), ('shows', (13, 18)), ('several', (19, 26)), ('tokenizer', (27, 36)), ('algorithms', (37, 47)), ('.', (47, 48))]
[('Hopefully', (0, 9)), (',', (9, 10)), ('you', (11, 14)), ('will', (15, 19)), ('be', (20, 22)), ('able', (23, 27)), ('to', (28, 30)), ('understand', (31, 41)), ('how', (42, 45)), ('they', (46, 50)), ('are', (51, 54)), ('trained', (55, 62)), ('and', (63, 66)), ('generate', (67, 75)), ('tokens', (76, 82)), ('.', (82, 83))]


In [20]:
from collections import defaultdict

word_freqs = defaultdict(int)
for text in corpus:
    words_with_offsets = tokenizer.backend_tokenizer.pre_tokenizer.pre_tokenize_str(text)
    new_words = [word for word, offset in words_with_offsets]
    for word in new_words:
        word_freqs[word] += 1

# Display word frequencies
for i, (word, freq) in enumerate(word_freqs.items()):
    if i == len(word_freqs) - 1:
        print(f" '{word}': {freq}")
    else:
        print(f" '{word}': {freq},")

 'This': 3,
 'is': 2,
 'the': 1,
 'Hugging': 1,
 'Face': 1,
 'Course': 1,
 '.': 4,
 'chapter': 1,
 'about': 1,
 'tokenization': 1,
 'section': 1,
 'shows': 1,
 'several': 1,
 'tokenizer': 1,
 'algorithms': 1,
 'Hopefully': 1,
 ',': 1,
 'you': 1,
 'will': 1,
 'be': 1,
 'able': 1,
 'to': 1,
 'understand': 1,
 'how': 1,
 'they': 1,
 'are': 1,
 'trained': 1,
 'and': 1,
 'generate': 1,
 'tokens': 1


As we saw before, the <font color='blue'>alphabet</font> is the <font color='blue'>unique set</font> composed of all the <font color='blue'>first letters</font> of <font color='blue'>words</font>, and <font color='blue'>all</font> the <font color='blue'>other letters</font> that appear in words <font color='blue'>prefixed</font> by <font color='blue'>`##`</font>:

In [21]:
alphabet = []
for word in word_freqs.keys():
    if word[0] not in alphabet:
        alphabet.append(word[0])
    for letter in word[1:]:
        if f"##{letter}" not in alphabet:
            alphabet.append(f"##{letter}")

alphabet.sort()

print(alphabet[:23])
print(alphabet[23:])

['##a', '##b', '##c', '##d', '##e', '##f', '##g', '##h', '##i', '##k', '##l', '##m', '##n', '##o', '##p', '##r', '##s', '##t', '##u', '##v', '##w', '##y', '##z']
[',', '.', 'C', 'F', 'H', 'T', 'a', 'b', 'c', 'g', 'h', 'i', 's', 't', 'u', 'w', 'y']


We also add the <font color='blue'>special tokens</font> used by the model at the <font color='blue'>beginning</font> of that <font color='blue'>vocabulary</font>.

In the case of BERT, it's the list `["[PAD]", "[UNK]", "[CLS]", "[SEP]", "[MASK]"]`:


In [22]:
vocab = ["[PAD]", "[UNK]", "[CLS]", "[SEP]", "[MASK]"] + alphabet.copy()

Next we need to <font color='blue'>split</font> each <font color='blue'>word</font>, with all the <font color='blue'>letters</font> that are <font color='blue'>not</font> the <font color='blue'>first prefixed</font> by <font color='blue'>`##`</font>:

In [23]:
splits = {
    word: [c if i == 0 else f"##{c}" for i, c in enumerate(word)]
    for word in word_freqs.keys()
}

In [24]:
for k, v in splits.items():
    print(k, v)

This ['T', '##h', '##i', '##s']
is ['i', '##s']
the ['t', '##h', '##e']
Hugging ['H', '##u', '##g', '##g', '##i', '##n', '##g']
Face ['F', '##a', '##c', '##e']
Course ['C', '##o', '##u', '##r', '##s', '##e']
. ['.']
chapter ['c', '##h', '##a', '##p', '##t', '##e', '##r']
about ['a', '##b', '##o', '##u', '##t']
tokenization ['t', '##o', '##k', '##e', '##n', '##i', '##z', '##a', '##t', '##i', '##o', '##n']
section ['s', '##e', '##c', '##t', '##i', '##o', '##n']
shows ['s', '##h', '##o', '##w', '##s']
several ['s', '##e', '##v', '##e', '##r', '##a', '##l']
tokenizer ['t', '##o', '##k', '##e', '##n', '##i', '##z', '##e', '##r']
algorithms ['a', '##l', '##g', '##o', '##r', '##i', '##t', '##h', '##m', '##s']
Hopefully ['H', '##o', '##p', '##e', '##f', '##u', '##l', '##l', '##y']
, [',']
you ['y', '##o', '##u']
will ['w', '##i', '##l', '##l']
be ['b', '##e']
able ['a', '##b', '##l', '##e']
to ['t', '##o']
understand ['u', '##n', '##d', '##e', '##r', '##s', '##t', '##a', '##n', '##d']
how ['h'

Now that we are ready for training, let's write a <font color='blue'>function</font> that <font color='blue'>computes</font> the <font color='blue'>score</font> of <font color='blue'>each pair</font>. We'll need to use this at each step of the training:

In [25]:
def compute_pair_scores(splits):
    """
    Computes the WordPiece score for every adjacent token pair:

        score(a, b) = freq(a, b) / (freq(a) * freq(b))

    Unlike BPE, this biases merges toward rare tokens that nearly always appear together.
    """
    # Track how often each individual token and adjacent token pairs appear in the corpus
    letter_freqs = defaultdict(int)
    pair_freqs = defaultdict(int)

    for word, freq in word_freqs.items():
        # Get the current tokenization of this word
        split = splits[word]

        if len(split) == 1:
            # Single-token words have no pairs; just accumulate the token frequency
            letter_freqs[split[0]] += freq
            continue

        # Sliding window of size 2 over the token list to extract all adjacent pairs
        for i in range(len(split) - 1):
            pair = (split[i], split[i + 1])

            # Accumulate the individual frequency of the left token in the pair
            letter_freqs[split[i]] += freq

            # Accumulate the corpus frequency for this adjacent pair
            pair_freqs[pair] += freq

        # The final token is not a left-hand token, accumulate its frequency separately
        letter_freqs[split[-1]] += freq

    scores = {
        pair: freq / (letter_freqs[pair[0]] * letter_freqs[pair[1]])
        for pair, freq in pair_freqs.items()
    }

    return scores

Let's have a look at a <font color='blue'>part</font> of <font color='blue'>this dictionary</font> after the initial splits:

In [26]:
pair_scores = compute_pair_scores(splits)
for i, key in enumerate(pair_scores.keys()):
    print(f"{key}: {pair_scores[key]}")
    if i >= 5:
        break

('T', '##h'): 0.125
('##h', '##i'): 0.03409090909090909
('##i', '##s'): 0.02727272727272727
('i', '##s'): 0.1
('t', '##h'): 0.03571428571428571
('##h', '##e'): 0.011904761904761904


Now, finding the <font color='blue'>pair</font> with the <font color='blue'>best score</font> only takes a quick <font color='blue'>loop</font>:

In [27]:
best_pair = ""
max_score = None
for pair, score in pair_scores.items():
    if max_score is None or max_score < score:
        best_pair = pair
        max_score = score

print(best_pair, max_score)

('a', '##b') 0.2


Or we could do the <font color='blue'>same thing</font> in a more <font color='blue'>Pythonic</font> way:

In [14]:
best_pair = max(pair_scores, key=pair_scores.get)
max_score = pair_scores[best_pair]

print(best_pair, max_score)

('a', '##b') 0.2


So the <font color='blue'>first merge</font> to learn is <font color='blue'>`('a', '##b') -> 'ab'`</font>, and we add `'ab'` to the vocabulary:


In [28]:
vocab.append("ab")

To continue, we need to apply that <font color='blue'>merge</font> in our <font color='blue'>`splits` dictionary</font>. Let's write another function for this:

In [29]:
def merge_pair(a, b, splits):
    """
    Merge all occurrences of adjacent pair (a, b) into a single token.

    The "##" continuation prefix is stripped from b before concatenation
    so the merged token accurately reflects its position within the word.
    """
    for word in word_freqs:
        # Get the current tokenization of this word
        split = splits[word]

        # Single-token words have no adjacent pairs, skip them
        if len(split) == 1:
            continue

        i = 0
        while i < len(split) - 1:
            # Check if the current token and the next token match our target pair
            if split[i] == a and split[i + 1] == b:

                # Strip the "##" continuation prefix from b before concatenating.
                # If b has no "##" prefix, use normal concatenation.
                merge = a + b[2:] if b.startswith("##") else a + b

                # Replace the two tokens with their merged form:
                # • split[:i]    — everything before the pair
                # • [merge]      — the new combined token
                # • split[i+2:]  — everything after the pair
                split = split[:i] + [merge] + split[i + 2:]

            else:
                # No match at this position, advance to the next token
                i += 1

        # Update the splits dictionary with the newly merged token list
        splits[word] = split

    return splits

And we can have a look at the <font color='blue'>result</font> of the <font color='blue'>first merge</font>:

In [30]:
splits = merge_pair("a", "##b", splits)
splits["about"]

['ab', '##o', '##u', '##t']

Now we have everything we need to <font color='blue'>loop</font> until we have <font color='blue'>learned all</font> the <font color='blue'>merges</font> we want. Let's aim for a vocab size of <font color='blue'>40</font>:

In [31]:
vocab_size = 40
while len(vocab) < vocab_size:
    scores = compute_pair_scores(splits)
    best_pair = max(scores, key=scores.get)
    splits = merge_pair(*best_pair, splits)
    a, b = best_pair
    vocab.append(a + (b[2:] if b.startswith("##") else b))


We can then look at the generated vocabulary:

In [33]:
print(vocab[:5])
print(vocab[5:28])
print(vocab[28:])

['[PAD]', '[UNK]', '[CLS]', '[SEP]', '[MASK]']
['##a', '##b', '##c', '##d', '##e', '##f', '##g', '##h', '##i', '##k', '##l', '##m', '##n', '##o', '##p', '##r', '##s', '##t', '##u', '##v', '##w', '##y', '##z']
[',', '.', 'C', 'F', 'H', 'T', 'a', 'b', 'c', 'g', 'h', 'i', 's', 't', 'u', 'w', 'y', 'ab']


As we can see, compared to BPE, this tokenizer learns <font color='blue'>parts of words</font> as <font color='blue'>tokens</font> a bit <font color='blue'>faster</font>.

To <font color='blue'>tokenize</font> a <font color='blue'>new text</font>, we <font color='blue'>pre-tokenize</font> it, <font color='blue'>split</font> it, then <font color='blue'>apply</font> the <font color='blue'>tokenization algorithm</font> on <font color='blue'>each word</font>. That is, we look for the <font color='blue'>biggest subword starting</font> at the <font color='blue'>beginning</font> of the <font color='blue'>first word</font> and <font color='blue'>split it</font>, then we <font color='blue'>repeat the process</font> on the second part, and so on for the <font color='blue'>rest</font> of <font color='blue'>that word and</font> the <font color='blue'>following words</font> in the text:


In [34]:
def encode_word(word):
    """
    Tokenize a word into WordPiece subword tokens via greedy longest-match-first.
    Remaining suffixes are prefixed with "##". Returns ["[UNK]"] if any substring
    has no vocab match.
    """
    tokens = []

    while len(word) > 0:
        # Shrink prefix until a vocab match is found
        i = len(word)
        while i > 0 and word[:i] not in vocab:
            i -= 1

        # No prefix matched — whole word is unrepresentable
        if i == 0:
            return ["[UNK]"]

        # Consume the longest matching prefix
        tokens.append(word[:i])
        word = word[i:]

        # Mark the remaining suffix as a continuation token
        if len(word) > 0:
            word = f"##{word}"

    return tokens

Let's test it on <font color='blue'>one word</font> that's in the <font color='blue'>vocabulary</font>, and <font color='blue'>another</font> that <font color='blue'>isn't</font>:

In [35]:
print(encode_word("Hugging"))
print(encode_word("HOgging"))

['H', '##u', '##g', '##g', '##i', '##n', '##g']
['[UNK]']


Now, let's write a <font color='blue'>function</font> that <font color='blue'>tokenizes</font> a <font color='blue'>text</font>:

In [36]:
def tokenize(text):
    pre_tokenize_result = tokenizer._tokenizer.pre_tokenizer.pre_tokenize_str(text)
    pre_tokenized_text = [word for word, offset in pre_tokenize_result]
    encoded_words = [encode_word(word) for word in pre_tokenized_text]
    return sum(encoded_words, [])

We can try it on <font color='blue'>any text</font>:

In [37]:
tokenize("This is the Hugging Face course!")

['T',
 '##h',
 '##i',
 '##s',
 'i',
 '##s',
 't',
 '##h',
 '##e',
 'H',
 '##u',
 '##g',
 '##g',
 '##i',
 '##n',
 '##g',
 'F',
 '##a',
 '##c',
 '##e',
 'c',
 '##o',
 '##u',
 '##r',
 '##s',
 '##e',
 '[UNK]']

In [38]:
def build_vocab_and_tokenize_wp(text, target_vocab_size):
    # Initialize splits with ## prefixes
    local_splits = {
        word: [c if i == 0 else f"##{c}" for i, c in enumerate(word)]
        for word in word_freqs.keys()
    }
    vocab = ["[PAD]", "[UNK]", "[CLS]", "[SEP]", "[MASK]"] + alphabet.copy()

    while len(vocab) < target_vocab_size:
        scores = compute_pair_scores(local_splits)
        if not scores:
            break

        # Get the highest score
        best_pair = max(scores, key=scores.get)

        # Merge and compute the new token (strip ## from right side)
        a, b = best_pair
        new_token = a + b[2:] if b.startswith("##") else a + b

        local_splits = merge_pair(a, b, local_splits)
        vocab.append(new_token)

    # Build a local vocab set for encode_word lookups
    local_vocab_set = set(vocab)

    def encode_word_local(word):
        """Greedy longest-match-first using the locally built vocab."""
        tokens = []
        while len(word) > 0:
            i = len(word)
            while i > 0 and word[:i] not in local_vocab_set:
                i -= 1
            if i == 0:
                return ["[UNK]"]
            tokens.append(word[:i])
            word = word[i:]
            if len(word) > 0:
                word = f"##{word}"
        return tokens

    # Pre-tokenize with BERT's pre-tokenizer, then encode each word
    pre_tokenized = [
        word for word, _ in
        tokenizer._tokenizer.pre_tokenizer.pre_tokenize_str(text)
    ]
    encoded = [encode_word_local(word) for word in pre_tokenized]
    tokens = sum(encoded, [])

    return tokens, len(vocab)

In [39]:
sentence = "The multifaceted adventurer was unapologetically reimagining his journey."

for size in [50, 100, 200, 500]:
    tokens, actual_size = build_vocab_and_tokenize_wp(sentence, size)
    print(f"Vocab {actual_size:>4}: ({len(tokens):>2} tokens) {tokens}")

Vocab   50: (39 tokens) ['T', '##h', '##e', '[UNK]', 'a', '##d', '##v', '##e', '##n', '##t', '##u', '##r', '##e', '##r', 'w', '##a', '##s', 'u', '##n', '##a', '##p', '##o', '##l', '##o', '##g', '##e', '##t', '##i', '##c', '##a', '##l', '##l', '##y', '[UNK]', 'h', '##i', '##s', '[UNK]', '.']
Vocab  100: (36 tokens) ['Th', '##e', '[UNK]', 'a', '##d', '##v', '##e', '##n', '##t', '##ur', '##e', '##r', 'w', '##a', '##s', 'u', '##n', '##a', '##p', '##o', '##l', '##o', '##g', '##e', '##t', '##i', '##c', '##a', '##ll', '##y', '[UNK]', 'h', '##i', '##s', '[UNK]', '.']
Vocab  161: (35 tokens) ['Th', '##e', '[UNK]', 'a', '##d', '##v', '##e', '##n', '##t', '##ur', '##e', '##r', 'w', '##a', '##s', 'un', '##a', '##p', '##o', '##l', '##o', '##g', '##e', '##t', '##i', '##c', '##a', '##ll', '##y', '[UNK]', 'h', '##i', '##s', '[UNK]', '.']
Vocab  161: (35 tokens) ['Th', '##e', '[UNK]', 'a', '##d', '##v', '##e', '##n', '##t', '##ur', '##e', '##r', 'w', '##a', '##s', 'un', '##a', '##p', '##o', '##l', '##o

In [40]:
bert = AutoTokenizer.from_pretrained("bert-base-cased")
tokens = bert.tokenize(sentence)
print(f"BERT  ({len(tokens)} tokens): {tokens}")

BERT  (19 tokens): ['The', 'multi', '##face', '##ted', 'adventure', '##r', 'was', 'un', '##ap', '##olo', '##get', '##ically', 're', '##ima', '##gin', '##ing', 'his', 'journey', '.']
